# TreeSearchPlayer を継承した簡易AIとランダム方策を対戦させる

TreeSearchPlayer は木探索でコマンドを選ぶ Player の基底クラス（詳細は
docs/api/README.md の TreeSearchPlayer 節を参照）。既定の評価関数（残りHP割合の差）の
ままでも動くが、ここでは evaluate() をオーバーライドして「相手を瀕死にできたか」を
加点する例を示す。AI開発ユースケースの発展形。

Google Colabで開いた場合は、まず次のセルで `jpoke` をインストールする
（ローカルで `pip install jpoke` 済みなら不要）。

In [ ]:
%pip install -q jpoke

In [ ]:
from __future__ import annotations

from jpoke import Battle
from jpoke.players import RandomPlayer
from jpoke.players.tree_search_player import TreeSearchPlayer

In [ ]:
class KOFocusedPlayer(TreeSearchPlayer):
    """相手を瀕死にできる手を優先する簡易AI（evaluate() の拡張例）。"""

    def evaluate(self, battle: Battle) -> float:
        # super().evaluate(battle) は継承元 TreeSearchPlayer の既定実装（未変更の
        # 評価関数）をそのまま呼び出す記法。オーバーライドしつつ親クラスの処理も
        # 活かしたいときに使う
        # 既定の評価（残りHP割合の差）に、相手の瀕死数をボーナスとして加える
        base = super().evaluate(battle)
        opponent_team = battle.get_team(battle.opponent(self))
        n_fainted = sum(1 for mon in opponent_team if mon.fainted)
        return base + n_fainted

In [ ]:
# max_plies/max_nodesの詳細は docs/api/README.md の TreeSearchPlayer コンストラクタ節を参照
ai_player = KOFocusedPlayer("TreeSearchAI", max_plies=1, max_nodes=50)
# じしん（じめんタイプ、威力100）は相手のピカチュウ（でんきタイプ、弱点）に
# 抜群が入り確実に瀕死にできるが、みずでっぽう（みずタイプ、威力40、等倍かつ
# カビゴンはみずタイプではないためSTABも乗らない）は威力が低く、急所に当たっても
# 1発で瀕死にできない。威力・タイプ相性の異なる技を混在させることで、
# 「相手を瀕死にできる技」を優先するKOFocusedPlayerの判断を観察しやすくする
ai_player.add_pokemon("カビゴン", move_names=["じしん", "みずでっぽう"])

random_player = RandomPlayer("RandomPlayer")
# カビゴン（じしんで弱点を突かれない）ではなく、じめんが弱点のピカチュウにする
random_player.add_pokemon("ピカチュウ", move_names=["でんきショック", "でんこうせっか"])

In [ ]:
battle = Battle(ai_player, random_player, seed=1)

max_turns = 20
battle.play_out(max_turns=max_turns)
winner = battle.winner
if winner is None:
    print(f"結果: 決着つかず（{max_turns}ターン経過）")
else:
    print(f"結果: {winner.username} 勝利（{battle.turn}ターン）")
print(f"直近ターンで TreeSearchAI が展開したノード数: {ai_player.nodes_expanded}")
battle.print_logs()

試してみよう: evaluate() に「残り技のPP」や「急所を受けた回数」など
別の要素を加点・減点として加えると、AIの手の選び方がどう変わるか観察できる。
max_plies を2以上に増やして探索の深さを変えると、手の選び方や
nodes_expanded（展開ノード数）がどう変わるかも比較できる